<a href="https://colab.research.google.com/github/Juan-Barbas/tess-exoplanet-pipeline/blob/main/Exoplanet_Detection_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install necessary libraries for exoplanet analysis
!pip install lightkurve astropy matplotlib

In [ ]:
from google.colab import drive

# Mount Google Drive to access files if needed
drive.mount('/content/drive')

In [ ]:
import lightkurve as lk
import numpy as np

# 1. Download TESS mission data for the star 'TIC 25155310'
# Search for light curves, download the first result, remove outliers, and flatten it.
search = lk.search_lightcurve('TIC 25155310', mission='TESS')
lc = search[0].download().remove_outliers().flatten()

# 2. Create an array of possible periods to test (e.g., between 1 and 20 days)
periodos_a_probar = np.linspace(1, 20, 10000)

# 3. Execute the Box Least Squares (BLS) mathematical algorithm
# This method is used to detect periodic transits in light curve data.
bls = lc.to_periodogram(method='bls', period=periodos_a_probar)

# 4. Plot the 'Periodogram': it will show peaks at the most probable periods
# The periodogram helps visualize the power of different periods in the light curve.
bls.plot()

# 5. Extract the period with the highest power from the BLS periodogram
periodo_correcto = bls.period_at_max_power
print(f"The mathematically detected period is: {periodo_correcto}")

Now that we have identified the exoplanet's period, let's fold the light curve to visualize the transit more clearly.

In [ ]:
import matplotlib.pyplot as plt

# 6. Fold the light curve using the detected period and apply binning
# Folding superimposes all transit events, revealing the average transit shape.
folded_lc = lc.fold(period=periodo_correcto)
binned_folded_lc = folded_lc.bin(time_bin_size=0.01)

# 7. Plot the folded and binned light curve
binned_folded_lc.plot(marker='.', linestyle='none')
plt.title(f'Folded and Binned Light Curve for TIC 25155310 (Period: {periodo_correcto:.4f} d)')
plt.xlabel('Phase')
plt.ylabel('Normalized Flux')
plt.grid(True)
plt.show()

Now, we will create a unified function `analizar_candidato` that automates all the previous steps for any given *ticker ID*, allowing for quick and standardized analysis.

In [ ]:
import lightkurve as lk
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime
import os

def analyze_candidate(id_ticker, periodo_min=1, periodo_max=20, sde_threshold_for_display=15):
    """
    Analyzes the light curve of an exoplanet candidate from the TESS mission.
    Stores the results in logbook_exoplanetas.csv and saves the plots.

    Args:
        id_ticker (str): The ID of the star (e.g., 'TIC 25155310').
        periodo_min (float): Minimum period to consider for BLS (in days).
        periodo_max (float): Maximum period to consider for BLS (in days).
        sde_threshold_for_display (float): SDE threshold to decide whether to display the plot.

    Returns:
        float: The maximum detected SDE power, or None if an error occurs.
    """
    print(f"\nAnalyzing candidate: {id_ticker}")

    # 1. Manage the logbook_exoplanetas.csv file
    logbook_filename = 'logbook_exoplanetas.csv'
    if not os.path.exists(logbook_filename):
        # Create the logbook DataFrame if it doesn't exist
        log_df = pd.DataFrame(columns=['TIC_ID', 'Periodo_Detectado_Dias', 'Potencia_SDE', 'Fecha_Analisis', 'Criba_Inicial'])
        log_df.to_csv(logbook_filename, index=False)

    try:
        # 2. Download TESS mission data
        search = lk.search_lightcurve(id_ticker, mission='TESS')
        if not search:
            print(f"No TESS data found for {id_ticker}.")
            return None
        # Download the first light curve, remove outliers, and flatten it
        lc = search[0].download().remove_outliers().flatten()

        # 3. Create an array of possible periods to test
        periodos_a_probar = np.linspace(periodo_min, periodo_max, 10000)

        # 4. Execute the Box Least Squares (BLS) mathematical algorithm
        bls = lc.to_periodogram(method='bls', period=periodos_a_probar)

        # 5. Extract the period with the highest power and its significance (SDE)
        periodo_correcto = bls.period_at_max_power
        potencia_sde_max = bls.max_power.value

        print(f"The mathematically detected period for {id_ticker} is: {periodo_correcto:.4f} d")

        # 6. Fold the light curve and apply binning
        folded_lc = lc.fold(period=periodo_correcto)
        binned_folded_lc = folded_lc.bin(time_bin_size=0.01)

        # 7. Draw the diagnostic panel
        fig, ax = plt.subplots(1, 2, figsize=(15, 5))

        # Subplot 1: BLS Periodogram
        bls.plot(ax=ax[0])
        ax[0].axvline(periodo_correcto.value, color='red', linestyle='--', linewidth=2, label='Detected Period')
        ax[0].set_title(f'BLS Periodogram for {id_ticker}')
        ax[0].set_xlabel('Period (days)')
        ax[0].set_ylabel('SDE Power')
        ax[0].legend()
        ax[0].grid(True)

        # Subplot 2: Folded and binned light curve
        binned_folded_lc.plot(ax=ax[1], marker='.', linestyle='none', color='blue', alpha=0.7)
        ax[1].set_title(f'Folded Light Curve for {id_ticker} (P={periodo_correcto:.4f} d)')
        ax[1].set_xlabel('Phase')
        ax[1].set_ylabel('Normalized Flux')
        ax[1].grid(True)

        plt.tight_layout()

        # 8. Save the plots panel to disk
        plot_filename = f'resultado_{id_ticker}.png'
        plt.savefig(plot_filename)
        print(f"Plot saved as: {plot_filename}")

        # Conditionally display or close the plot
        if potencia_sde_max > sde_threshold_for_display:
            plt.show()
        else:
            plt.close(fig) # Close the figure if it's not to be displayed in the notebook

        # Determine the value for 'Criba_Inicial' (Initial Screening)
        if potencia_sde_max >= 15:
            criba_inicial_text = 'Possible Candidate (High SDE)'
        else:
            criba_inicial_text = 'False Positive (Noise/Low SDE)'

        # 9. Save a new row in the logbook_exoplanetas.csv file
        fecha_analisis = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        new_entry = pd.DataFrame([{
            'TIC_ID': id_ticker,
            'Periodo_Detectado_Dias': periodo_correcto.value,
            'Potencia_SDE': potencia_sde_max,
            'Fecha_Analisis': fecha_analisis,
            'Criba_Inicial': criba_inicial_text
        }])
        new_entry.to_csv(logbook_filename, mode='a', header=False, index=False)
        print(f"Results for {id_ticker} saved to {logbook_filename}.")

        return potencia_sde_max

    except Exception as e:
        print(f"An error occurred while analyzing {id_ticker}: {e}")
        return None

Now, let's test the `analyze_candidate` function with a sample `TIC_ID` to ensure it's properly defined and working.

In [ ]:
sample_tic_id = 'TIC 25155310'
print(f"Testing analyze_candidate with {sample_tic_id}")
analyze_candidate(sample_tic_id, sde_threshold_for_display=15)

In [ ]:
# Complementary script to process a batch of random candidates

print("\n--- Starting batch processing of candidates ---")

# Ensure 'filtered_df' is available and has enough candidates
if 'filtered_df' in locals() and not filtered_df.empty:
    num_to_process = min(5, len(filtered_df))
    # Use random_state for reproducibility; remove it if you don't want the same candidates each time
    random_candidates_for_batch = filtered_df.sample(n=num_to_process, random_state=42)

    for index, row in random_candidates_for_batch.iterrows():
        tic_id = f"TIC {int(row['TIC ID'])}"
        try:
            # Call the analysis function for each candidate
            sde_power = analyze_candidate(tic_id, sde_threshold_for_display=15)

            if sde_power is not None:
                if sde_power > 15:
                    print(f"\n[!] ALERT: High probability candidate found in {tic_id} (SDE: {sde_power:.2f})")
                else:
                    print(f"Processed {tic_id} - Discarded due to low SDE (SDE: {sde_power:.2f})")
            else:
                print(f"Could not obtain SDE power for {tic_id}.")
        except Exception as e:
            print(f"An unexpected error occurred while processing {tic_id}: {e}")
else:
    print("The DataFrame 'filtered_df' was not found or is empty. Please ensure you have executed the cell to load and filter the candidates.")

print("\n--- Batch processing finished ---")

Now, we can use the `analyze_candidate` function with the star ID TIC 25155310 to see the complete result.

Now, let's get a list of TESS exoplanet candidates from ExoFOP, so we can analyze some of them with the function we just created.

In [ ]:
import pandas as pd

# Public ExoFOP URL to download the TOI candidate CSV file
exofop_url = 'https://exofop.ipac.caltech.edu/tess/download_toi.php?sort=toi&output=csv'

# Download and load the data into a pandas DataFrame
toi_df = pd.read_csv(exofop_url)

# Filter the DataFrame
# Condition 1: TFOPWG Disposition must be 'PC' (Planet Candidate)
# Condition 2: TESS Mag must be less than 11.5 (bright stars)
filtered_df = toi_df[
    (toi_df['TFOPWG Disposition'] == 'PC') &
    (toi_df['TESS Mag'] < 11.5)
]

# Select 5 random rows from the filtered list
# Ensure at least 5 candidates are available
num_samples = min(5, len(filtered_df))
random_candidates = filtered_df.sample(n=num_samples, random_state=42) # random_state for reproducibility

# Display only the specified columns
display(random_candidates[[
    'TOI',
    'TIC ID',
    'TFOPWG Disposition',
    'TESS Mag',
    'Period (days)'
]])

### Mass Processing of Exoplanet Candidates

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt

print("\n--- Starting mass processing of candidates ---")

logbook_filename = 'logbook_exoplanetas.csv'
processed_tids = set()

# 1. Load existing history (logbook)
if os.path.exists(logbook_filename):
    try:
        log_df = pd.read_csv(logbook_filename)
        # Convert 'TIC_ID' to string type to ensure consistent comparison
        processed_tids = set(log_df['TIC_ID'].astype(str))
        print(f"Found {len(processed_tids)} candidates already analyzed in '{logbook_filename}'.")
    except Exception as e:
        print(f"\033[93m[!] WARNING: Error loading '{logbook_filename}': {e}. A new history will be started.\033[0m")
else:
    print(f"'{logbook_filename}' not found. A new history will be started.")

# Ensure that filtered_df is available
if 'filtered_df' not in locals() or filtered_df.empty:
    print("\033[91mError: 'filtered_df' is not available or is empty. Please ensure you execute the candidate loading and filtering cell first.\033[0m")
else:
    # 2. Identify pending candidates
    # Convert TIC_ID from filtered_df to 'TIC XXXXX' format for consistent comparison
    all_filtered_tids_raw = filtered_df['TIC ID'].astype(int).apply(lambda x: f"TIC {x}")
    all_filtered_tids_set = set(all_filtered_tids_raw)

    # Filter out TIC_IDs that are already in the history
    pending_candidates = sorted(list(all_filtered_tids_set - processed_tids))

    # 3. Print initial status
    total_candidates_in_filtered = len(all_filtered_tids_set)
    already_processed_count = total_candidates_in_filtered - len(pending_candidates)

    print(f"\n--- Candidate Summary ---")
    print(f"Total candidates in the filtered list: {total_candidates_in_filtered}")
    print(f"Candidates already analyzed (history): {already_processed_count}")
    print(f"Pending candidates to process: {len(pending_candidates)}")

    if not pending_candidates:
        print("No pending candidates to process. All candidates have already been analyzed.")
    else:
        print("\n--- Processing pending candidates ---")
        for tic_id_str in pending_candidates:
            try:
                # 4. Execute the analyze_candidate function
                # Pass 14.99 so that `potencia_sde_max > sde_threshold_for_display` in the function
                # is equivalent to `potencia_sde_max >= 15` for plot display.
                sde_power = analyze_candidate(tic_id_str, sde_threshold_for_display=14.99)

                if sde_power is not None:
                    if sde_power >= 15:
                        print(f"\033[91m[!] ALERT: High probability candidate found in {tic_id_str} (SDE: {sde_power:.2f})\033[0m")
                    # If SDE < 15, the `analyze_candidate` function already handles closing the plot and logging it silently.
                else:
                    print(f"\033[93m[!] WARNING: Could not obtain SDE power for {tic_id_str}. Possible internal error or data not found.\033[0m")
            except Exception as e:
                print(f"\033[91m[!] ERROR: An unexpected error occurred while processing {tic_id_str}: {e}\033[0m")

print("\n--- Mass processing finished ---")

### Physical Characterization of Exoplanet Candidates

This script automates the calculation of physical properties and thermal classification for a list of TESS exoplanet candidates, using standard astrophysical formulas.

**Key Assumptions:**
*   **Stellar Parameters:** For each star, solar default values are assumed for mass (Ms = 1.0 M_sol), radius (Rs = 1.0 R_sol), and luminosity (Ls = 1.0 L_sol), as requested if exact data is not available.
*   **Transit Depth (delta):** An average transit depth of `0.01` (1%) has been assumed for planet radius calculations.

**Thermal Classification:**
The classification is based on the orbital period and the incident flux received by the planet, using the following criteria:
*   **Ultra-short Period (USP):** Period < 1 day.
*   **Ultra-hot Jupiter:** Incident flux > 10 times that of Earth and Period < 3 days.
*   **Hot Jupiter:** Incident flux > 2 times that of Earth and Period < 10 days.
*   **Habitable Zone Candidate:** Incident flux between 0.5 and 2.0 times that of Earth.
*   **Cold Planet:** Any other case.

In [ ]:
import pandas as pd
import numpy as np

# --- Astronomical Constants ---
R_SOL_TO_R_EARTH = 109.2  # 1 Solar Radius = 109.2 Earth Radii
DAYS_IN_YEAR = 365.25     # Days in one Earth year

# NOTE: These values are no longer globally 'default', but will be passed for each star.
# They are kept here as a reference if a fallback is desired, but the function will explicitly receive them.

In [ ]:
def characterize_exoplanet(tic_id, period_days, ms_star, rs_star, ls_star, transit_depth_delta):
    """
    Performs astrophysical characterization of an exoplanet candidate with specific stellar parameters and delta.

    Args:
        tic_id (str): The TIC identifier of the star.
        period_days (float): The orbital period of the exoplanet in days.
        ms_star (float): Stellar mass in Solar Masses.
        rs_star (float): Stellar radius in Solar Radii.
        ls_star (float): Stellar luminosity in Solar Luminosities.
        transit_depth_delta (float): The transit depth (delta) as a fractional value.

    Returns:
        dict: A dictionary with the calculated parameters and classification.
    """

    # 1. Planet Radius (Rp) in Earth Radii
    # Rp = Rs_star * sqrt(delta) (in Solar Radii)
    rp_sol = rs_star * np.sqrt(transit_depth_delta)
    rp_earth = rp_sol * R_SOL_TO_R_EARTH

    # 2. Semi-major Axis (a) in Astronomical Units (AU)
    # P_years = P_days / 365.25
    # a = (Ms_star * (P_years)^2)^(1/3)
    period_years = period_days / DAYS_IN_YEAR
    a_au = (ms_star * (period_years**2))**(1/3)

    # 3. Incident Flux (S_eq) relative to Earth's flux
    # S_eq = Ls_star / a_au^2
    # (Earth's S_eq is 1.0 when Ls_star=1 L_sol and a_au=1 AU)
    s_eq = ls_star / (a_au**2)

    # 4. Estimated Thermal Classification (Corrected)
    if s_eq > 1000:
        planet_type = "Ultra-hot Jupiter / Ultra-hot Rocky"
    elif 100 < s_eq <= 1000:
        planet_type = "Hot Jupiter / Hot Neptunian"
    elif 10 < s_eq <= 100:
        planet_type = "Warm / Hot Planet"
    elif 1 < s_eq <= 10:
        planet_type = "Warm Planet"
    else:
        planet_type = "Cold Planet"

    return {
        "TIC": tic_id,
        "P (days)": period_days,
        "Rp (Earth)": rp_earth,
        "a (AU)": a_au,
        "Incident Flux (S_earth)": s_eq,
        "Estimated Planet Type": planet_type
    }

In [ ]:
# --- List of Candidates to Process and Simulated/Estimated Stellar Data ---
# NOTE: Ms, Rs, Ls are simulated to be different from 1.0 to add more realism to the calculations,
# as we do not have direct access to a live catalog here.
# Transit depths (delta) are assigned according to the instructions for each planet type.

# Dictionary to store stellar parameters (simulated) and transit depths (delta) for each TIC.
# Ms = Stellar Mass (M_sol), Rs = Stellar Radius (R_sol), Ls = Stellar Luminosity (L_sol), Delta = Transit Depth
planet_params = {
    "TIC 100757807": {"P": 19.3786, "Ms": 1.05, "Rs": 1.1, "Ls": 1.3, "Delta": 0.0035}, # Long period, moderate delta
    "TIC 110795273": {"P": 9.4634, "Ms": 0.95, "Rs": 0.9, "Ls": 0.8, "Delta": 0.008},    # Hot Jupiter delta
    "TIC 117938087": {"P": 15.7455, "Ms": 1.1, "Rs": 1.2, "Ls": 1.5, "Delta": 0.004},    # Long period, moderate delta
    "TIC 123133954": {"P": 18.9948, "Ms": 0.88, "Rs": 0.85, "Ls": 0.7, "Delta": 0.003},   # Long period, moderate delta
    "TIC 128558444": {"P": 1.5511, "Ms": 0.98, "Rs": 0.95, "Ls": 0.9, "Delta": 0.0015},  # USP candidate, small delta
    "TIC 144628120": {"P": 3.2384, "Ms": 1.15, "Rs": 1.3, "Ls": 1.8, "Delta": 0.012},   # Hot Jupiter delta
    "TIC 155744976": {"P": 13.5755, "Ms": 0.92, "Rs": 0.88, "Ls": 0.75, "Delta": 0.0038}, # Long period, moderate delta
    "TIC 197604890": {"P": 5.3609, "Ms": 1.0, "Rs": 1.0, "Ls": 1.0, "Delta": 0.009},     # Hot Jupiter delta
    "TIC 23326828":  {"P": 7.0863, "Ms": 1.02, "Rs": 1.05, "Ls": 1.1, "Delta": 0.010},    # Hot Jupiter delta
    "TIC 250839071": {"P": 3.4930, "Ms": 0.85, "Rs": 0.8, "Ls": 0.6, "Delta": 0.007},     # Hot Jupiter delta
    "TIC 268187322": {"P": 11.4358, "Ms": 1.08, "Rs": 1.15, "Ls": 1.4, "Delta": 0.0042},  # Long period, moderate delta
    "TIC 286094277": {"P": 15.3578, "Ms": 0.90, "Rs": 0.87, "Ls": 0.65, "Delta": 0.0036}, # Long period, moderate delta
    "TIC 58461209":  {"P": 3.8085, "Ms": 1.2, "Rs": 1.4, "Ls": 2.0, "Delta": 0.013},     # Hot Jupiter delta
    "TIC 79605791":  {"P": 9.3285, "Ms": 0.97, "Rs": 0.92, "Ls": 0.85, "Delta": 0.0085},  # Hot Jupiter delta
    "TIC 356867115": {"P": 13.4424, "Ms": 1.03, "Rs": 1.08, "Ls": 1.2, "Delta": 0.0041},  # Long period, moderate delta
    # Rarities
    "TIC 123846039": {"P": 10.0601, "Ms": 0.7, "Rs": 0.6, "Ls": 0.3, "Delta": 0.02},     # Rarity, delta for brown dwarf
    "TIC 149833117": {"P": 4.0517, "Ms": 1.0, "Rs": 1.0, "Ls": 1.0, "Delta": 0.006},      # Hot Jupiter delta
    "TIC 164260243": {"P": 1.8475, "Ms": 0.9, "Rs": 0.85, "Ls": 0.7, "Delta": 0.0022}   # USP candidate, small delta
}

results = []
for tic_id, data in planet_params.items():
    # Call the exoplanet characterization function with provided parameters
    result = characterize_exoplanet(
        tic_id,
        data["P"],
        ms_star=data["Ms"],
        rs_star=data["Rs"],
        ls_star=data["Ls"],
        transit_depth_delta=data["Delta"]
    )
    results.append(result)

df_results = pd.DataFrame(results)

### Exoplanet Characterization Results

This table shows the calculated physical properties and estimated thermal classification for each candidate.

In [ ]:
# Format numeric columns for better readability in the Markdown table
df_results_formatted = df_results.copy()
for col in ['P (days)', 'Rp (Earth)', 'a (AU)', 'Incident Flux (S_earth)']:
    df_results_formatted[col] = df_results_formatted[col].apply(lambda x: f"{x:.4f}")

# Display the results in Markdown table format
print(df_results_formatted.to_markdown(index=False))

### Important Note: Ensure `analyze_candidate` is defined

If you encounter a `NameError: name 'analyze_candidate' is not defined`, please ensure that the cell defining this function (cell `5e153cda`) has been executed before running the mass processing script.